# Hugging Face Tabular 모델 벤치마킹

In [26]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. 환경 설정

먼저, 각 모델에 필요한 라이브러리를 설치합니다. 여기에는 TabPFN을 위한 `tabpfn`, AutoGluon을 위한 `autogluon.tabular[all]`, 일반적인 Hugging Face 모델 상호작용을 위한 `transformers`, 그리고 데이터 처리 및 평가를 위한 기타 유틸리티가 포함됩니다.

In [27]:
pip install tabpfn autogluon.tabular[all] transformers datasets scikit-learn pandas -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
synthefy-nori 0.12.1 requires huggingface-hub>=1.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [28]:
from huggingface_hub import hf_hub_download
print("huggingface hub ok")

huggingface hub ok


In [29]:
pip install synthefy-nori -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.57.6 requires huggingface-hub<1.0,>=0.34.0, but you have huggingface-hub 1.26.0 which is incompatible.


In [30]:
import subprocess

# 1) tabpfn_v2: model_path="Prior-Labs/TabPFN-v2-reg"는 tabpfn 라이브러리가 인식하는
#    체크포인트 이름이 아니라서 다운로드가 실패한다. 기본 체크포인트를 쓰도록 고친다.
#    (이미 수정된 상태일 수 있으므로 idempotent하게 처리)
path = "/content/drive/MyDrive/Colab Notebooks/benchmark.py"
src = open(path, encoding="utf-8").read()

old = (
    "            return TabPFNRegressor(\n"
    "                model_path=\"Prior-Labs/TabPFN-v2-reg\",\n"
    "                random_state=RANDOM_STATE,\n"
    "            )\n"
)
new = (
    "            return TabPFNRegressor(\n"
    "                random_state=RANDOM_STATE,\n"
    "            )\n"
)
count = src.count(old)
if count == 0:
    print("tabpfn_v2 model_path: 버그 패턴이 없음 (이미 수정된 상태) — 건너뜀")
else:
    assert count == 1, f"expected 0 or 1 match, got {count}"
    src = src.replace(old, new)
    open(path, "w", encoding="utf-8").write(src)
    print("tabpfn_v2 model_path 버그 수정 완료")

# 2) nori: synthefy_nori 패키지가 설치돼 있는지 확인, 없으면 설치
try:
    import synthefy_nori
    print("synthefy_nori 이미 설치됨:", synthefy_nori.__file__)
except ImportError:
    result = subprocess.run(
        ["pip", "install", "synthefy-nori", "-q"],
        capture_output=True, text=True,
    )
    print("\n--- pip install synthefy-nori ---")
    print("returncode:", result.returncode)
    print(result.stdout[-1500:])
    print(result.stderr[-1500:])


tabpfn_v2 model_path: 버그 패턴이 없음 (이미 수정된 상태) — 건너뜀
synthefy_nori 이미 설치됨: /usr/local/lib/python3.12/dist-packages/synthefy_nori/__init__.py


In [35]:
path = "/content/drive/MyDrive/Colab Notebooks/benchmark.py"
src = open(path, encoding="utf-8").read()

old = (
    "            return TabPFNRegressor(\n"
    "                random_state=RANDOM_STATE,\n"
    "            )\n"
)
new = (
    "            return TabPFNRegressor(\n"
    "                model_path=\"tabpfn-v2-regressor-v2_default.ckpt\",\n"
    "                random_state=RANDOM_STATE,\n"
    "            )\n"
)
count = src.count(old)

if count == 0:
    print("tabpfn_v2 model_path: 버그 패턴이 없음 (이미 수정된 상태) — 건너뜀")
else:
    assert count == 1, f"expected 1 match, got {count}"
    src = src.replace(old, new)
    open(path, "w", encoding="utf-8").write(src)
    print("model_path를 Prior-Labs/TabPFN-v2-reg 내 체크포인트 파일명으로 명시 완료")

tabpfn_v2 model_path: 버그 패턴이 없음 (이미 수정된 상태) — 건너뜀


## 2. 모델 설명 및 실행

이제 각 모델을 로드하고 기본적인 예측을 시연합니다. 벤치마킹을 위해서는 더미 데이터를 실제 데이터셋으로 교체해야 합니다.

### 2-1. Prior-Labs/TabPFN-v2-reg

이 모델은 회귀 작업을 위한 Tabular 사전 학습형 신경망입니다. `tabpfn` 라이브러리를 활용합니다.

In [37]:
path = "/content/drive/MyDrive/Colab Notebooks/benchmark.py"
src = open(path, encoding="utf-8").read()

old = (
    "        if self.name == \"tabpfn_v2\":\n"
    "            try:\n"
    "                from tabpfn import TabPFNRegressor\n"
    "            except ImportError as exc:\n"
    "                raise RuntimeError(\n"
    "                    \"TabPFN 실행에는 `pip install tabpfn`이 필요합니다.\"\n"
    "                ) from exc\n"
    "            return TabPFNRegressor(\n"
    "                model_path=\"tabpfn-v2-regressor-v2_default.ckpt\",\n"
    "                random_state=RANDOM_STATE,\n"
    "            )\n"
)
new = (
    "        if self.name == \"tabpfn_v2\":\n"
    "            try:\n"
    "                from tabpfn import TabPFNRegressor\n"
    "            except ImportError as exc:\n"
    "                raise RuntimeError(\n"
    "                    \"TabPFN 실행에는 `pip install tabpfn`이 필요합니다.\"\n"
    "                ) from exc\n"
    "            from huggingface_hub import hf_hub_download\n"
    "            local_ckpt = hf_hub_download(\n"
    "                repo_id=\"Prior-Labs/TabPFN-v2-reg\",\n"
    "                filename=\"tabpfn-v2-regressor-v2_default.ckpt\",\n"
    "            )\n"
    "            return TabPFNRegressor(\n"
    "                model_path=local_ckpt,\n"
    "                random_state=RANDOM_STATE,\n"
    "            )\n"
)
count = src.count(old)

if count == 0:
    print("tabpfn_v2: HF 다운로드 패턴이 없음 (이미 수정된 상태) — 건너뜀")
else:
    assert count == 1, f"expected 1 match, got {count}"
    src = src.replace(old, new)
    open(path, "w", encoding="utf-8").write(src)
    print("tabpfn_v2: HF에서 직접 로컬로 받아서 로딩하도록 수정 완료")

tabpfn_v2: HF 다운로드 패턴이 없음 (이미 수정된 상태) — 건너뜀


In [38]:
import json
from pathlib import Path
import pandas as pd

MODEL = "tabpfn_v2"
artifact_dir = Path("/content/drive/MyDrive/Colab Notebooks") / f"colab_{MODEL}"

!python "/content/drive/MyDrive/Colab Notebooks/benchmark.py" benchmark \
    --data "/content/drive/MyDrive/Colab Notebooks/sizekorea_measurements_clean.csv" \
    --models {MODEL} \
    --artifact-dir "{artifact_dir}" \
    --height 170 \
    --weight 65

metrics_path = artifact_dir / "metrics.json"
if metrics_path.exists():
    df = pd.DataFrame(json.loads(metrics_path.read_text(encoding="utf-8")))
    display(df)
else:
    print(f"{MODEL}: metrics.json이 생성되지 않았습니다 — 위 실행 로그의 에러를 확인하세요.")


tabpfn-v2-regressor-v2_default.ckpt: downloading bytes:  76% 33.7M/44.4M [00:01<00:00, 40.6MB/s, 1.79MB/s  ]
tabpfn-v2-regressor-v2_default.ckpt: downloading bytes: 100% 41.3M/41.3M [00:01<00:00, 32.6MB/s, 3.99MB/s  ]
tabpfn-v2-regressor-v2_default.ckpt: reconstructing file: 100% 44.4M/44.4M [00:01<00:00, 35.1MB/s, 4.32MB/s  ]
    model  mean_mae  mean_rmse  mean_r2  mean_p90_error  fit_seconds  predict_ms_per_row
tabpfn_v2  1.815097   2.331424 0.807721        3.718603    10.829658            10.02315

입력값 비교 예측
{
  "input": {
    "gender": "M",
    "height": 170.0,
    "weight": 65.0
  },
  "predictions": {
    "tabpfn_v2": {
      "chest": 96.0,
      "waist": 80.8,
      "hip": 92.3,
      "thigh": 54.5,
      "calf": 36.3,
      "arm": 31.3,
      "shoulder": 39.0
    }
  }
}

상세 지표: /content/drive/MyDrive/Colab Notebooks/colab_tabpfn_v2/metrics.json


,model,target,mae,rmse,r2,p90_absolute_error,fit_seconds,predict_ms_per_row
0,tabpfn_v2,chest,2.298802,2.929496,0.886822,4.615457,10.829658,10.02315
1,tabpfn_v2,waist,3.202583,4.077546,0.838454,6.495128,10.829658,10.02315
2,tabpfn_v2,hip,1.942027,2.500641,0.831306,3.945862,10.829658,10.02315
3,tabpfn_v2,thigh,2.034358,2.644842,0.712360,4.293206,10.829658,10.02315
4,tabpfn_v2,calf,1.109889,1.432779,0.781108,2.359315,10.829658,10.02315
5,tabpfn_v2,arm,1.061246,1.395656,0.829405,2.123804,10.829658,10.02315
6,tabpfn_v2,shoulder,1.056776,1.339012,0.774592,2.197445,10.829658,10.02315


### 2-2. Synthefy/Nori

Nori는 트랜스포머 기반의 테이블형 모델입니다. `benchmark.py`는 `synthefy_nori` 패키지의 `NoriRegressor`를 사용합니다.

In [39]:
import json
from pathlib import Path
import pandas as pd

MODEL = "nori"
artifact_dir = Path("/content/drive/MyDrive/Colab Notebooks") / f"colab_{MODEL}"

!python "/content/drive/MyDrive/Colab Notebooks/benchmark.py" benchmark \
    --data "/content/drive/MyDrive/Colab Notebooks/sizekorea_measurements_clean.csv" \
    --models {MODEL} \
    --artifact-dir "{artifact_dir}" \
    --height 170 \
    --weight 65

metrics_path = artifact_dir / "metrics.json"
if metrics_path.exists():
    df = pd.DataFrame(json.loads(metrics_path.read_text(encoding="utf-8")))
    display(df)
else:
    print(f"{MODEL}: metrics.json이 생성되지 않았습니다 — 위 실행 로그의 에러를 확인하세요.")


nori.pt: downloading bytes:  94% 44.7M/47.3M [00:00<00:00, 74.9MB/s, 2.52MB/s  ]
nori.pt: downloading bytes: 100% 44.7M/44.7M [00:01<00:00, 42.1MB/s, 4.35MB/s  ]
nori.pt: reconstructing file: 100% 47.3M/47.3M [00:01<00:00, 44.6MB/s, 4.62MB/s  ]
config.json: 100% 549/549 [00:00<00:00, 1.82MB/s]
model  mean_mae  mean_rmse  mean_r2  mean_p90_error  fit_seconds  predict_ms_per_row
 nori  1.827325   2.342642 0.805508        3.730246     2.171632           30.725025

입력값 비교 예측
{
  "input": {
    "gender": "M",
    "height": 170.0,
    "weight": 65.0
  },
  "predictions": {
    "nori": {
      "chest": 96.3,
      "waist": 80.5,
      "hip": 92.4,
      "thigh": 54.5,
      "calf": 36.8,
      "arm": 31.4,
      "shoulder": 38.9
    }
  }
}

상세 지표: /content/drive/MyDrive/Colab Notebooks/colab_nori/metrics.json


,model,target,mae,rmse,r2,p90_absolute_error,fit_seconds,predict_ms_per_row
0,nori,chest,2.308623,2.932379,0.886599,4.662309,2.171632,30.725025
1,nori,waist,3.214036,4.082610,0.838053,6.452791,2.171632,30.725025
2,nori,hip,1.966272,2.521668,0.828457,3.977149,2.171632,30.725025
3,nori,thigh,2.065957,2.682930,0.704016,4.366920,2.171632,30.725025
4,nori,calf,1.115867,1.438895,0.779236,2.330606,2.171632,30.725025
5,nori,arm,1.060315,1.395688,0.829396,2.107088,2.171632,30.725025
6,nori,shoulder,1.060205,1.344323,0.772801,2.214859,2.171632,30.725025


### 2-3. AutoGluon TabPFNMix Regressor

이 모델은 회귀를 위한 사전 학습된 AutoGluon 예측기입니다. AutoGluon의 `TabularPredictor`는 Hugging Face에 저장된 모델을 직접 로드할 수 있습니다.

In [43]:
print('Attempting to fix huggingface-hub dependency issues...')

# Uninstall current huggingface-hub to resolve conflicts
!pip uninstall -y huggingface-hub

# Reinstall all necessary packages, allowing pip to resolve dependencies
# It's important to reinstall packages that showed conflicts or depend on huggingface-hub
!pip install tabpfn autogluon.tabular[all] transformers datasets scikit-learn pandas synthefy-nori -q

Attempting to fix huggingface-hub dependency issues...
Found existing installation: huggingface_hub 1.26.0
Uninstalling huggingface_hub-1.26.0:
  Successfully uninstalled huggingface_hub-1.26.0
Requested datasets from https://files.pythonhosted.org/packages/46/1a/b9f9b3bfef624686ae81c070f0a6bb635047b17cdb3698c7ad01281e6f9a/datasets-1.6.2-py3-none-any.whl has invalid metadata: Expected matching RIGHT_PARENTHESIS for LEFT_PARENTHESIS, after version specifier
    pyarrow (>=1.0.0<4.0.0)
            ~~~~~~~~^
Please use pip<24.1 if you need to use this version.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Now that the dependencies should be resolved, let's re-run the `tabpfn_mix` benchmark to generate its metrics and prediction files.

In [44]:
import json
from pathlib import Path
import pandas as pd

MODEL = "tabpfn_mix"
artifact_dir = Path("/content/drive/MyDrive/Colab Notebooks") / f"colab_{MODEL}"

# Ensure the artifact directory exists, in case the benchmark doesn't create it reliably
artifact_dir.mkdir(parents=True, exist_ok=True)

print(f"Running benchmark for {MODEL}...")
!python "/content/drive/MyDrive/Colab Notebooks/benchmark.py" benchmark \
    --data "/content/drive/MyDrive/Colab Notebooks/sizekorea_measurements_clean.csv" \
    --models {MODEL} \
    --artifact-dir "{artifact_dir}" \
    --height 170 \
    --weight 65

metrics_path = artifact_dir / "metrics.json"
if metrics_path.exists():
    df = pd.DataFrame(json.loads(metrics_path.read_text(encoding="utf-8")))
    display(df)
else:
    print(f"{MODEL}: metrics.json이 생성되지 않았습니다 — 위 실행 로그의 에러를 확인하세요.")

Running benchmark for tabpfn_mix...
		No module named 'huggingface_hub'
Traceback (most recent call last):
  File "/content/drive/MyDrive/Colab Notebooks/benchmark.py", line 935, in <module>
    main()
  File "/content/drive/MyDrive/Colab Notebooks/benchmark.py", line 884, in main
    metrics, sample_predictions = benchmark(
                                  ^^^^^^^^^^
  File "/content/drive/MyDrive/Colab Notebooks/benchmark.py", line 600, in benchmark
    model.fit(x_train, y_train)
  File "/content/drive/MyDrive/Colab Notebooks/benchmark.py", line 271, in fit
    predictor.fit(
  File "/usr/local/lib/python3.12/dist-packages/autogluon/common/utils/decorators.py", line 34, in _call
    return f(*gargs, **gkwargs)
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/autogluon/tabular/predictor/predictor.py", line 1412, in fit
    self._fit(ag_fit_kwargs=ag_fit_kwargs, ag_post_fit_kwargs=ag_post_fit_kwargs)
  File "/usr/local/lib/python3.12/dist-packages/autog

,model,target,mae,rmse,r2,p90_absolute_error,fit_seconds,predict_ms_per_row
0,tabpfn_mix,chest,2.546508,3.238158,0.861005,5.200912,261.19747,3.469721
1,tabpfn_mix,waist,3.367777,4.202849,0.828560,6.599998,261.19747,3.469721
2,tabpfn_mix,hip,2.434030,3.140833,0.732470,5.100002,261.19747,3.469721
3,tabpfn_mix,thigh,2.301391,2.948656,0.641016,4.757554,261.19747,3.469721
4,tabpfn_mix,calf,1.134439,1.462404,0.772405,2.400001,261.19747,3.469721
5,tabpfn_mix,arm,1.099175,1.452997,0.814521,2.200000,261.19747,3.469721
6,tabpfn_mix,shoulder,1.173557,1.482008,0.723138,2.400002,261.19747,3.469721


With `tabpfn_mix` hopefully having run successfully, let's re-run the comparison cell to see the combined results from all models.

In [ ]:
import json
from pathlib import Path
import pandas as pd

MODELS = ["tabpfn_v2", "nori", "tabpfn_mix"]
base_dir = Path("/content/drive/MyDrive/Colab Notebooks")

detail_frames = []
for model in MODELS:
    metrics_path = base_dir / f"colab_{model}" / "metrics.json"
    if metrics_path.exists():
        df_model_metrics = pd.DataFrame(json.loads(metrics_path.read_text(encoding="utf-8")))
        # Add a 'model' column to each individual model's metrics for easier concatenation
        df_model_metrics['model'] = model
        detail_frames.append(df_model_metrics)
    else:
        print(f"{model}: metrics.json이 없습니다 — 2번 섹션에서 먼저 실행하세요.")

if detail_frames:
    detail_df = pd.concat(detail_frames, ignore_index=True)
    # The previous `c2485998` was trying to group by 'model' on the 'metrics' column directly.
    # The `metrics.json` content has columns like 'mae', 'rmse', etc. not 'mae' itself as a column.
    # So, the original grouping was incorrect in assuming 'mae' was a list or a directly groupable item.
    # I will modify this to correctly aggregate the mean of the metrics for each model.

    # First, let's display the combined detail_df for inspection
    print("Combined detailed metrics:")
    display(detail_df)

    # Now, let's create the summary_df by grouping and aggregating
    summary_df = (
        detail_df.groupby("model", as_index=False)[['mae', 'rmse', 'r2', 'p90_absolute_error', 'fit_seconds', 'predict_ms_per_row']]
        .mean() # Calculate the mean for each metric
        .sort_values("mae") # Sort by mae for consistent ordering
    )

    print("\nSummary of model comparisons:")
    display(summary_df.round(4))
else:
    print("비교할 결과가 없습니다.")

Let's confirm that all the required `test_set_` and `test_predictions_` CSV files have been generated for each model in their respective directories within your Google Drive.

In [ ]:
from pathlib import Path

MODELS = ["tabpfn_v2", "nori", "tabpfn_mix"]
base_dir = Path("/content/drive/MyDrive/Colab Notebooks")

for model in MODELS:
    model_dir = base_dir / f"colab_{model}"
    print(f"\nListing files for model: {model}")
    if model_dir.exists():
        for file_path in model_dir.iterdir():
            print(f"- {file_path.name}")
        # You can add more specific checks here if needed, e.g., for specific file names
        # expected_set_file = f"test_set_{model}.csv"
        # expected_pred_file = f"test_predictions_{model}.csv"
        # if (model_dir / expected_set_file).exists():
        #     print(f"  '{expected_set_file}' found.")
        # else:
        #     print(f"  '{expected_set_file}' NOT found.")
    else:
        print(f"  Directory '{model_dir.name}' not found. Model benchmark might not have completed.")

### 1. Reinstall All Dependencies (after runtime restart)

This step ensures all required libraries are installed with compatible versions, resolving any previous dependency conflicts.

In [46]:
!pip install tabpfn huggingface-hub transformers datasets scikit-learn pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 86.8 MB/s eta 0:00:00


In [47]:
!pip install synthefy-nori -q

In [50]:
!pip install "autogluon.tabular[tabpfnmix]"

### 2. Re-apply `benchmark.py` Patches

After a runtime restart, changes to `/content/` files are reset. These cells re-apply the necessary modifications to `benchmark.py`.

In [51]:
import subprocess

# 1) tabpfn_v2: model_path="Prior-Labs/TabPFN-v2-reg"는 tabpfn 라이브러리가 인식하는
#    체크포인트 이름이 아니라서 다운로드가 실패한다. 기본 체크포인트를 쓰도록 고친다.
#    (이미 수정된 상태일 수 있으므로 idempotent하게 처리)
path = "/content/drive/MyDrive/Colab Notebooks/benchmark.py"
src = open(path, encoding="utf-8").read()

old = (
    "            return TabPFNRegressor(\n"
    "                model_path=\"Prior-Labs/TabPFN-v2-reg\",\n"
    "                random_state=RANDOM_STATE,\n"
    "            )\n"
)
new = (
    "            return TabPFNRegressor(\n"
    "                random_state=RANDOM_STATE,\n"
    "            )\n"
)
count = src.count(old)
if count == 0:
    print("tabpfn_v2 model_path: 버그 패턴이 없음 (이미 수정된 상태) — 건너뜀")
else:
    assert count == 1, f"expected 0 or 1 match, got {count}"
    src = src.replace(old, new)
    open(path, "w", encoding="utf-8").write(src)
    print("tabpfn_v2 model_path 버그 수정 완료")

# 2) nori: synthefy_nori 패키지가 설치돼 있는지 확인, 없으면 설치
# (This part is not strictly necessary after consolidated install, but kept for idempotency)
try:
    import synthefy_nori
    print("synthefy_nori 이미 설치됨:", synthefy_nori.__file__)
except ImportError:
    result = subprocess.run(
        ["pip", "install", "synthefy-nori", "-q"],
        capture_output=True, text=True,
    )
    print("\n--- pip install synthefy-nori ---")
    print("returncode:", result.returncode)
    print(result.stdout[-1500:])
    print(result.stderr[-1500:])

tabpfn_v2 model_path: 버그 패턴이 없음 (이미 수정된 상태) — 건너뜀
synthefy_nori 이미 설치됨: /usr/local/lib/python3.12/dist-packages/synthefy_nori/__init__.py


In [52]:
path = "/content/drive/MyDrive/Colab Notebooks/benchmark.py"
src = open(path, encoding="utf-8").read()

old = (
    "            return TabPFNRegressor(\n"
    "                random_state=RANDOM_STATE,\n"
    "            )\n"
)
new = (
    "            return TabPFNRegressor(\n"
    "                model_path=\"tabpfn-v2-regressor-v2_default.ckpt\",\n"
    "                random_state=RANDOM_STATE,\n"
    "            )\n"
)
count = src.count(old)

if count == 0:
    print("tabpfn_v2 model_path: 버그 패턴이 없음 (이미 수정된 상태) — 건너뜀")
else:
    assert count == 1, f"expected 1 match, got {count}"
    src = src.replace(old, new)
    open(path, "w", encoding="utf-8").write(src)
    print("model_path를 Prior-Labs/TabPFN-v2-reg 내 체크포인트 파일명으로 명시 완료")

tabpfn_v2 model_path: 버그 패턴이 없음 (이미 수정된 상태) — 건너뜀


In [53]:
path = "/content/drive/MyDrive/Colab Notebooks/benchmark.py"
src = open(path, encoding="utf-8").read()

old = (
    "        if self.name == \"tabpfn_v2\":\n"
    "            try:\n"
    "                from tabpfn import TabPFNRegressor\n"
    "            except ImportError as exc:\n"
    "                raise RuntimeError(\n"
    "                    \"TabPFN 실행에는 `pip install tabpfn`이 필요합니다.\"\n"
    "                ) from exc\n"
    "            return TabPFNRegressor(\n"
    "                model_path=\"tabpfn-v2-regressor-v2_default.ckpt\",\n"
    "                random_state=RANDOM_STATE,\n"
    "            )\n"
)
new = (
    "        if self.name == \"tabpfn_v2\":\n"
    "            try:\n"
    "                from tabpfn import TabPFNRegressor\n"
    "            except ImportError as exc:\n"
    "                raise RuntimeError(\n"
    "                    \"TabPFN 실행에는 `pip install tabpfn`이 필요합니다.\"\n"
    "                ) from exc\n"
    "            from huggingface_hub import hf_hub_download\n"
    "            local_ckpt = hf_hub_download(\n"
    "                repo_id=\"Prior-Labs/TabPFN-v2-reg\",\n"
    "                filename=\"tabpfn-v2-regressor-v2_default.ckpt\",\n"
    "            )\n"
    "            return TabPFNRegressor(\n"
    "                model_path=local_ckpt,\n"
    "                random_state=RANDOM_STATE,\n"
    "            )\n"
)
count = src.count(old)

if count == 0:
    print("tabpfn_v2: HF 다운로드 패턴이 없음 (이미 수정된 상태) — 건너뜀")
else:
    assert count == 1, f"expected 1 match, got {count}"
    src = src.replace(old, new)
    open(path, "w", encoding="utf-8").write(src)
    print("tabpfn_v2: HF에서 직접 로컬로 받아서 로딩하도록 수정 완료")

tabpfn_v2: HF 다운로드 패턴이 없음 (이미 수정된 상태) — 건너뜀


### 3. Run Benchmarks for Each Model

Running each model's benchmark. Note that `--height` and `--weight` parameters are removed as per your request; the `benchmark.py` script will now use the entire `sizekorea_measurements_clean.csv` for evaluation.

In [54]:
import json
from pathlib import Path
import pandas as pd

MODEL = "tabpfn_v2"
artifact_dir = Path("/content/drive/MyDrive/Colab Notebooks") / f"colab_{MODEL}"
artifact_dir.mkdir(parents=True, exist_ok=True)

print(f"Running benchmark for {MODEL}...")
!python "/content/drive/MyDrive/Colab Notebooks/benchmark.py" benchmark \
    --data "/content/drive/MyDrive/Colab Notebooks/sizekorea_measurements_clean.csv" \
    --models {MODEL} \
    --artifact-dir "{artifact_dir}"

metrics_path = artifact_dir / "metrics.json"
if metrics_path.exists():
    df = pd.DataFrame(json.loads(metrics_path.read_text(encoding="utf-8")))
    display(df)
else:
    print(f"{MODEL}: metrics.json이 생성되지 않았습니다 — 위 실행 로그의 에러를 확인하세요.")

Running benchmark for tabpfn_v2...
    model  mean_mae  mean_rmse  mean_r2  mean_p90_error  fit_seconds  predict_ms_per_row
tabpfn_v2  1.815097   2.331424 0.807721        3.718603     6.558312           10.348557

입력값 비교 예측
{
  "input": {
    "gender": "M",
    "height": 170.0,
    "weight": 65.0
  },
  "predictions": {
    "tabpfn_v2": {
      "chest": 96.0,
      "waist": 80.8,
      "hip": 92.3,
      "thigh": 54.5,
      "calf": 36.3,
      "arm": 31.3,
      "shoulder": 39.0
    }
  }
}

상세 지표: /content/drive/MyDrive/Colab Notebooks/colab_tabpfn_v2/metrics.json


,model,target,mae,rmse,r2,p90_absolute_error,fit_seconds,predict_ms_per_row
0,tabpfn_v2,chest,2.298802,2.929496,0.886822,4.615457,6.558312,10.348557
1,tabpfn_v2,waist,3.202583,4.077546,0.838454,6.495128,6.558312,10.348557
2,tabpfn_v2,hip,1.942027,2.500641,0.831306,3.945862,6.558312,10.348557
3,tabpfn_v2,thigh,2.034358,2.644842,0.712360,4.293206,6.558312,10.348557
4,tabpfn_v2,calf,1.109889,1.432779,0.781108,2.359315,6.558312,10.348557
5,tabpfn_v2,arm,1.061246,1.395656,0.829405,2.123804,6.558312,10.348557
6,tabpfn_v2,shoulder,1.056776,1.339012,0.774592,2.197445,6.558312,10.348557


In [ ]:
import json
from pathlib import Path
import pandas as pd

MODEL = "nori"
artifact_dir = Path("/content/drive/MyDrive/Colab Notebooks") / f"colab_{MODEL}"
artifact_dir.mkdir(parents=True, exist_ok=True)

print(f"Running benchmark for {MODEL}...")
!python "/content/drive/MyDrive/Colab Notebooks/benchmark.py" benchmark \
    --data "/content/drive/MyDrive/Colab Notebooks/sizekorea_measurements_clean.csv" \
    --models {MODEL} \
    --artifact-dir "{artifact_dir}"

metrics_path = artifact_dir / "metrics.json"
if metrics_path.exists():
    df = pd.DataFrame(json.loads(metrics_path.read_text(encoding="utf-8")))
    display(df)
else:
    print(f"{MODEL}: metrics.json이 생성되지 않았습니다 — 위 실행 로그의 에러를 확인하세요.")

In [ ]:
import json
from pathlib import Path
import pandas as pd

MODEL = "tabpfn_mix"
artifact_dir = Path("/content/drive/MyDrive/Colab Notebooks") / f"colab_{MODEL}"
artifact_dir.mkdir(parents=True, exist_ok=True)

print(f"Running benchmark for {MODEL}...")
!python "/content/drive/MyDrive/Colab Notebooks/benchmark.py" benchmark \
    --data "/content/drive/MyDrive/Colab Notebooks/sizekorea_measurements_clean.csv" \
    --models {MODEL} \
    --artifact-dir "{artifact_dir}"

metrics_path = artifact_dir / "metrics.json"
if metrics_path.exists():
    df = pd.DataFrame(json.loads(metrics_path.read_text(encoding="utf-8")))
    display(df)
else:
    print(f"{MODEL}: metrics.json이 생성되지 않았습니다 — 위 실행 로그의 에러를 확인하세요.")

### 4. Consolidated Model Comparison

Here's a summary of the performance metrics for all models.

In [ ]:
import json
from pathlib import Path
import pandas as pd

MODELS = ["tabpfn_v2", "nori", "tabpfn_mix"]
base_dir = Path("/content/drive/MyDrive/Colab Notebooks")

detail_frames = []
for model in MODELS:
    metrics_path = base_dir / f"colab_{model}" / "metrics.json"
    if metrics_path.exists():
        df_model_metrics = pd.DataFrame(json.loads(metrics_path.read_text(encoding="utf-8")))
        df_model_metrics['model'] = model # Add a 'model' column for easier concatenation
        detail_frames.append(df_model_metrics)
    else:
        print(f"{model}: metrics.json이 없습니다 — 3번 섹션에서 먼저 실행하세요.")

if detail_frames:
    detail_df = pd.concat(detail_frames, ignore_index=True)
    print("Combined detailed metrics:")
    display(detail_df)

    summary_df = (
        detail_df.groupby("model", as_index=False)[['mae', 'rmse', 'r2', 'p90_absolute_error', 'fit_seconds', 'predict_ms_per_row']]
        .mean() # Calculate the mean for each metric
        .sort_values("mae") # Sort by mae for consistent ordering
    )

    print("\nSummary of model comparisons:")
    display(summary_df.round(4))
else:
    print("비교할 결과가 없습니다.")

### 5. Verify Output Files

Confirming the generation of `test_set_*.csv` and `test_predictions_*.csv` files for each model.

In [ ]:
from pathlib import Path

MODELS = ["tabpfn_v2", "nori", "tabpfn_mix"]
base_dir = Path("/content/drive/MyDrive/Colab Notebooks")

for model in MODELS:
    model_dir = base_dir / f"colab_{model}"
    print(f"\nListing files for model: {model}")
    if model_dir.exists():
        files_found = list(model_dir.iterdir())
        if files_found:
            for file_path in files_found:
                print(f"- {file_path.name}")
            # More specific checks for test_set_*.csv and test_predictions_*.csv
            expected_set_file = f"test_set_{model}.csv"
            expected_pred_file = f"test_predictions_{model}.csv"
            if (model_dir / expected_set_file).exists():
                print(f"  '{expected_set_file}' found.")
            else:
                print(f"  '{expected_set_file}' NOT found. Please check benchmark execution for {model}.")
            if (model_dir / expected_pred_file).exists():
                print(f"  '{expected_pred_file}' found.")
            else:
                print(f"  '{expected_pred_file}' NOT found. Please check benchmark execution for {model}.")
        else:
            print(f"  No files found in directory '{model_dir.name}'.")
    else:
        print(f"  Directory '{model_dir.name}' not found. Model benchmark might not have completed.")

In [41]:
import json
from pathlib import Path
import pandas as pd

MODEL = "tabpfn_mix"
artifact_dir = Path("/content/drive/MyDrive/Colab Notebooks") / f"colab_{MODEL}"

!python "/content/drive/MyDrive/Colab Notebooks/benchmark.py" benchmark \
    --data "/content/drive/MyDrive/Colab Notebooks/sizekorea_measurements_clean.csv" \
    --models {MODEL} \
    --artifact-dir "{artifact_dir}" \
    --height 170 \
    --weight 65

metrics_path = artifact_dir / "metrics.json"
if metrics_path.exists():
    df = pd.DataFrame(json.loads(metrics_path.read_text(encoding="utf-8")))
    display(df)
else:
    print(f"{MODEL}: metrics.json이 생성되지 않았습니다 — 위 실행 로그의 에러를 확인하세요.")

config.json: 100% 169/169 [00:00<00:00, 940kB/s]

model.safetensors: downloading bytes:  95% 148M/156M [00:01<00:00, 182MB/s, 12.4MB/s  ]
model.safetensors: downloading bytes: 100% 148M/148M [00:01<00:00, 80.1MB/s, 13.9MB/s  ]
model.safetensors: reconstructing file: 100% 156M/156M [00:01<00:00, 84.5MB/s, 14.9MB/s  ]
^C


,model,target,mae,rmse,r2,p90_absolute_error,fit_seconds,predict_ms_per_row
0,tabpfn_mix,chest,2.546508,3.238158,0.861005,5.200912,261.19747,3.469721
1,tabpfn_mix,waist,3.367777,4.202849,0.828560,6.599998,261.19747,3.469721
2,tabpfn_mix,hip,2.434030,3.140833,0.732470,5.100002,261.19747,3.469721
3,tabpfn_mix,thigh,2.301391,2.948656,0.641016,4.757554,261.19747,3.469721
4,tabpfn_mix,calf,1.134439,1.462404,0.772405,2.400001,261.19747,3.469721
5,tabpfn_mix,arm,1.099175,1.452997,0.814521,2.200000,261.19747,3.469721
6,tabpfn_mix,shoulder,1.173557,1.482008,0.723138,2.400002,261.19747,3.469721


## 3. 모델 비교

모델을 효과적으로 비교하려면 공통 테스트 데이터셋(특징 및 실제 레이블)이 필요합니다. 여기서는 더미 데이터를 사용하여 RMSE(Root Mean Squared Error) 및 MAE(Mean Absolute Error)와 같은 메트릭을 계산하는 방법을 시연합니다.

In [57]:
import json
from pathlib import Path
import pandas as pd

MODELS = ["tabpfn_v2", "nori", "tabpfn_mix"]
base_dir = Path("/content/drive/MyDrive/Colab Notebooks")

detail_frames = []
for model in MODELS:
    metrics_path = base_dir / f"colab_{model}" / "metrics.json"
    if metrics_path.exists():
        detail_frames.append(pd.DataFrame(json.loads(metrics_path.read_text(encoding="utf-8"))))
    else:
        print(f"{model}: metrics.json이 없습니다 — 2번 섹션에서 먼저 실행하세요.")

if detail_frames:
    detail_df = pd.concat(detail_frames, ignore_index=True)
    summary_df = (
        detail_df.groupby("model", as_index=False)
        .agg(
            mean_mae=("mae", "mean"),
            mean_rmse=("rmse", "mean"),
            mean_r2=("r2", "mean"),
            mean_p90_error=("p90_absolute_error", "mean"),
            fit_seconds=("fit_seconds", "max"),
            predict_ms_per_row=("predict_ms_per_row", "max"),
        )
        .sort_values("mean_mae")
    )
    display(summary_df.round(4))
else:
    print("비교할 결과가 없습니다.")

,model,mean_mae,mean_rmse,mean_r2,mean_p90_error,fit_seconds,predict_ms_per_row
2,tabpfn_v2,1.8151,2.3314,0.8077,3.7186,6.5583,10.3486
0,nori,1.8273,2.3426,0.8055,3.7302,2.1716,30.7250
1,tabpfn_mix,2.0081,2.5611,0.7676,4.0941,261.1975,3.4697


In [58]:
import shutil
from pathlib import Path

base_dir = Path("/content/drive/MyDrive/Colab Notebooks")
for name in ["colab_tabpfn_v2", "colab_nori", "colab_tabpfn_mix"]:
    d = base_dir / name
    if d.exists():
        shutil.rmtree(d)
        print(f"삭제됨: {d}")
    else:
        print(f"없음(이미 정리됨): {d}")


삭제됨: /content/drive/MyDrive/Colab Notebooks/colab_tabpfn_v2
삭제됨: /content/drive/MyDrive/Colab Notebooks/colab_nori
삭제됨: /content/drive/MyDrive/Colab Notebooks/colab_tabpfn_mix


In [59]:
from pathlib import Path
base = Path("/content/drive/MyDrive/Colab Notebooks")

bp = base / "benchmark.py"
ts = base / "test_set.csv"

src = bp.read_text(encoding="utf-8")
print("benchmark.py 크기:", bp.stat().st_size, "bytes")
print("source_row_id 있음:", "ROW_ID" in src and 'ROW_ID = "source_row_id"' in src)
print("--test-data 옵션 있음:", "--test-data" in src)

import pandas as pd
df = pd.read_csv(ts)
print("\ntest_set.csv 행 수:", len(df))
print("컬럼:", df.columns.tolist())
print(df.head(2))


benchmark.py 크기: 33498 bytes
source_row_id 있음: True
--test-data 옵션 있음: True

test_set.csv 행 수: 1000
컬럼: ['source_row_id', 'gender', 'height', 'weight', 'actual_chest', 'actual_waist', 'actual_hip', 'actual_thigh', 'actual_calf', 'actual_arm', 'actual_shoulder']
   source_row_id gender  height  weight  actual_chest  actual_waist  \
0            996      M   175.5    84.7         105.6          90.2   
1           3481      F   147.7    51.2          81.5          76.4   

   actual_hip  actual_thigh  actual_calf  actual_arm  actual_shoulder  
0       103.9          63.4         39.7        35.5             40.5  
1        89.2          52.2         33.7        27.9             33.6  


In [60]:
MODEL = "tabpfn_v2"
artifact_dir = "/content/drive/MyDrive/Colab Notebooks/_work_" + MODEL

!python "/content/drive/MyDrive/Colab Notebooks/benchmark.py" benchmark \
    --data "/content/drive/MyDrive/Colab Notebooks/sizekorea_measurements_clean.csv" \
    --models {MODEL} \
    --test-data "/content/drive/MyDrive/Colab Notebooks/test_set.csv" \
    --artifact-dir "{artifact_dir}"


    model  mean_mae  mean_rmse  mean_r2  mean_p90_error  fit_seconds  predict_ms_per_row
tabpfn_v2  1.815507   2.331854 0.807634        3.720932       7.2303            9.614189

입력값 비교 예측
{
  "input": {
    "gender": "M",
    "height": 170.0,
    "weight": 65.0
  },
  "predictions": {
    "tabpfn_v2": {
      "chest": 96.1,
      "waist": 80.9,
      "hip": 92.4,
      "thigh": 54.5,
      "calf": 36.2,
      "arm": 31.3,
      "shoulder": 39.0
    }
  }
}

상세 지표: /content/drive/MyDrive/Colab Notebooks/_work_tabpfn_v2/metrics.json


In [61]:
MODEL = "nori"
artifact_dir = "/content/drive/MyDrive/Colab Notebooks/_work_" + MODEL

!python "/content/drive/MyDrive/Colab Notebooks/benchmark.py" benchmark \
    --data "/content/drive/MyDrive/Colab Notebooks/sizekorea_measurements_clean.csv" \
    --models {MODEL} \
    --test-data "/content/drive/MyDrive/Colab Notebooks/test_set.csv" \
    --artifact-dir "{artifact_dir}"


model  mean_mae  mean_rmse  mean_r2  mean_p90_error  fit_seconds  predict_ms_per_row
 nori  1.829106   2.344426 0.805365        3.733241     1.646679           26.245669

입력값 비교 예측
{
  "input": {
    "gender": "M",
    "height": 170.0,
    "weight": 65.0
  },
  "predictions": {
    "nori": {
      "chest": 96.3,
      "waist": 80.6,
      "hip": 92.4,
      "thigh": 54.5,
      "calf": 36.7,
      "arm": 31.5,
      "shoulder": 38.9
    }
  }
}

상세 지표: /content/drive/MyDrive/Colab Notebooks/_work_nori/metrics.json


In [62]:
MODEL = "tabpfn_mix"
artifact_dir = "/content/drive/MyDrive/Colab Notebooks/_work_" + MODEL

!python "/content/drive/MyDrive/Colab Notebooks/benchmark.py" benchmark \
    --data "/content/drive/MyDrive/Colab Notebooks/sizekorea_measurements_clean.csv" \
    --models {MODEL} \
    --test-data "/content/drive/MyDrive/Colab Notebooks/test_set.csv" \
    --artifact-dir "{artifact_dir}"


Traceback (most recent call last):
  File "/content/drive/MyDrive/Colab Notebooks/benchmark.py", line 973, in <module>
    main()
  File "/content/drive/MyDrive/Colab Notebooks/benchmark.py", line 917, in main
    metrics, sample_predictions = benchmark(
                                  ^^^^^^^^^^
  File "/content/drive/MyDrive/Colab Notebooks/benchmark.py", line 630, in benchmark
    model.fit(x_train, y_train)
  File "/content/drive/MyDrive/Colab Notebooks/benchmark.py", line 272, in fit
    predictor.fit(
  File "/usr/local/lib/python3.12/dist-packages/autogluon/common/utils/decorators.py", line 34, in _call
    return f(*gargs, **gkwargs)
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/autogluon/tabular/predictor/predictor.py", line 1412, in fit
    self._fit(ag_fit_kwargs=ag_fit_kwargs, ag_post_fit_kwargs=ag_post_fit_kwargs)
  File "/usr/local/lib/python3.12/dist-packages/autogluon/tabular/predictor/predictor.py", line 1420, in _fit
    self._post_

In [64]:
import shutil
import time
import torch
from pathlib import Path

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

import sys
sys.path.insert(0, "/content/drive/MyDrive/Colab Notebooks")
import importlib
import benchmark as bm
importlib.reload(bm)

import pandas as pd
frame = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/sizekorea_measurements_clean.csv")
data = bm.validate_frame(frame)
test = bm.validate_frame(
    bm.load_test_frame(Path("/content/drive/MyDrive/Colab Notebooks/test_set.csv"))
)
train = data[~data[bm.ROW_ID].isin(test[bm.ROW_ID])].copy()

x_train = bm.make_model_features(train)
y_train = train[bm.TARGETS]
print(f"train rows: {len(train)}, test rows: {len(test)}")

from autogluon.tabular import TabularPredictor

target = "chest"
train_data = x_train.copy()
train_data[target] = y_train[target].to_numpy()

hyperparameters = {
    "TABPFNMIX": [{
        "model_path_classifier": "autogluon/tabpfn-mix-1.0-classifier",
        "model_path_regressor": "autogluon/tabpfn-mix-1.0-regressor",
        "n_ensembles": 1,
        "max_epochs": 2,
    }]
}

diag_path = "/content/tabpfnmix_diag"
shutil.rmtree(diag_path, ignore_errors=True)

print(f"[{time.strftime('%H:%M:%S')}] fit 시작...")
predictor = TabularPredictor(
    label=target,
    problem_type="regression",
    path=diag_path,
    verbosity=3,
)
predictor.fit(
    train_data=train_data,
    hyperparameters=hyperparameters,
    raise_on_no_models_fitted=False,
)
print(f"[{time.strftime('%H:%M:%S')}] fit 종료")
print("\n=== fit_summary ===")
print(predictor.fit_summary(verbosity=3))


CUDA available: True
GPU: Tesla T4
train rows: 4092, test rows: 1000


Verbosity: 3 (Detailed Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.56/14.56 GB
Total GPU Memory:   Free: 14.56 GB, Allocated: 0.00 GB, Total: 14.56 GB
GPU Count:          1
Memory Avail:       10.68 GB / 12.67 GB (84.3%)
Disk Space Avail:   58.28 GB / 112.64 GB (51.7%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : New in v1.5: The state-of-the-art for tabular data. Massively better than 'best' on datasets <100000 samples by using new Tabular Foundation Models (T

[02:21:55] fit 시작...


Beginning AutoGluon training ...
AutoGluon will save models to "/content/tabpfnmix_diag"
Train Data Rows:    4092
Train Data Columns: 3
Label Column:       chest
Problem Type:       regression
Preprocessing data ...
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    10934.40 MB
	Train Data (Original)  Memory Usage: 0.09 MB (0.0% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
			Note: Converting 1 features to boolean dtype as they only contain 2 unique values.
			Original Features (exact raw dtype, raw dtype):
				('float64', 'float') : 3 | ['gender', 'height', 'weight']
			Types of features in original data (raw dtype, special dtypes):
				('float', []) : 3 | ['gender', 'height', 'weight']
			Types of features in processed data (raw dtype, s

[02:24:35] fit 종료

=== fit_summary ===
*** Summary of fit() ***
Estimated performance of each model:
                 model  score_val              eval_metric  pred_time_val    fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0            TabPFNMix  -3.051188  root_mean_squared_error      19.266363  139.368418               19.266363         139.368418            1       True          1
1  WeightedEnsemble_L2  -3.051188  root_mean_squared_error      19.267065  139.380999                0.000703           0.012581            2       True          2
Number of models trained: 2
Types of models trained:
{'WeightedEnsembleModel', 'TabPFNMixModel'}
Bagging used: False 
Multi-layer stack-ensembling used: False 
Feature Metadata (Processed):
(raw dtype, special dtypes):
('float', [])     : 2 | ['height', 'weight']
('int', ['bool']) : 1 | ['gender']
Plot summary of models saved to file: /content/tabpfnmix_diag/SummaryOfModels.html
*** End of fit() summary 

In [66]:
from pathlib import Path
bp = Path("/content/drive/MyDrive/Colab Notebooks/benchmark.py")
src = bp.read_text(encoding="utf-8")
print("크기:", bp.stat().st_size, "bytes")
print("num_gpus 코드 있음:", "num_gpus" in src and "torch.cuda.is_available()" in src)

idx = src.find("import torch")
print(src[idx-50:idx+500])


크기: 33708 bytes
num_gpus 코드 있음: True
max_epochs": 30,
            }]
        }
        import torch

        use_gpu = torch.cuda.is_available()
        for target in TARGETS:
            train_data = x.copy()
            train_data[target] = y[target].to_numpy()
            predictor = TabularPredictor(
                label=target,
                problem_type="regression",
                path=str(self._temp_dir / target),
                verbosity=0,
            )
            predictor.fit(
                train_data=train_data,
                hyperparameters=hyperparameters,


In [ ]:
MODEL = "tabpfn_mix"
artifact_dir = "/content/drive/MyDrive/Colab Notebooks/_work_" + MODEL

!python "/content/drive/MyDrive/Colab Notebooks/benchmark.py" benchmark \
    --data "/content/drive/MyDrive/Colab Notebooks/sizekorea_measurements_clean.csv" \
    --models {MODEL} \
    --test-data "/content/drive/MyDrive/Colab Notebooks/test_set.csv" \
    --artifact-dir "{artifact_dir}"
